In [ ]:
# === Report column headers ===

#import pandas as pd
#df = pd.read_csv("/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/y_pred_rfc_long.csv")  # or the merged-with-features file
#df.columns.tolist()

In [1]:
# Read in results data and report shape

import pandas as pd

RFC_PATH = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/y_pred_rfc_long.csv"
MLP_PATH = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/y_pred_mlp_long.csv"

rfc = pd.read_csv(RFC_PATH)
mlp = pd.read_csv(MLP_PATH)

print(rfc.shape, mlp.shape)
print(rfc.head(3))

(4389, 7) (4389, 7)
     TokenID language    paradigm    mode gold pred  correct
0   42005026      eng  participle  binary    1    1     True
1   42022032      eng  participle  binary    1    1     True
2  43002010b      eng  participle  binary    1    1     True


In [2]:
# Read in X_dev and merge
XDEV_PATH = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/X_dev.csv"

xdev = pd.read_csv(XDEV_PATH)

# Merge features onto predictions
rfcX = rfc.merge(xdev, on="TokenID", how="left")
mlpX = mlp.merge(xdev, on="TokenID", how="left")

print(rfcX.shape)
print("Missing feature rows:", rfcX.filter(regex="^Dt|^TL|^Cl|^Vc|^AS|^Akt").isna().all(axis=1).sum())

(4389, 25)
Missing feature rows: 0


In [3]:
FEATURE_COLS = [
    "DtExp", "DtRes", "DtUniv", "DtDTAM",
    "TLTS", "TLAspPfv", "TLMdIrr",
    "ClNegPol", "ClIntgv", "ClCntfcl",
    "VcPass",
    "ASTrns", "ASUnrg", "ASUncc",
    "AktState", "AktAct", "AktAcc", "AktAch"
]

BASE_COLS = ["TokenID", "language", "paradigm", "mode", "gold", "pred", "correct"]

REPORT_COLS = BASE_COLS + FEATURE_COLS

In [4]:
import os
from datetime import datetime

BASE_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/Reports"

# Timestamped run folder so you never overwrite anything accidentally
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")
OUT_DIR = os.path.join(BASE_DIR, RUN_ID)
os.makedirs(OUT_DIR, exist_ok=True)

# Set up export
def save_csv(df, name):
    path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"Saved: {path} ({len(df)} rows)")

print("Writing reports to:", OUT_DIR)

Writing reports to: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/Reports/20260122_1659


In [5]:
def filter_preds(df, language=None, paradigm=None, mode=None, gold=None, pred=None, correct=None, features=None):
    out = df.copy()

    # Standard column filters (AND logic)
    if language is not None:
        out = out[out["language"] == language]
    if paradigm is not None:
        out = out[out["paradigm"] == paradigm]
    if mode is not None:
        out = out[out["mode"] == mode]
    if gold is not None:
        out = out[out["gold"] == gold]
    if pred is not None:
        out = out[out["pred"] == pred]
    if correct is not None:
        out = out[out["correct"] == correct]

    # Feature filters (also AND logic)
    if features is not None:
        for col, val in features.items():
            out = out[out[col] == val]

    return out

In [6]:
# Gold = le, split by correctness
le_correct = filter_preds(rfcX, language="cmn", paradigm="le_guo", gold="le", correct=True)
le_wrong   = filter_preds(rfcX, language="cmn", paradigm="le_guo", gold="le", correct=False)

# Compare feature rates (mean = proportion of 1s) for a few features
feat_cols = ["DtExp", "DtRes", "DtUniv", "DtDTAM", "AktState", "AktAct", "AktAcc", "AktAch"]

summary = pd.DataFrame({
    "correct_mean": le_correct[feat_cols].mean(numeric_only=True),
    "wrong_mean":   le_wrong[feat_cols].mean(numeric_only=True),
})
summary["diff_correct_minus_wrong"] = summary["correct_mean"] - summary["wrong_mean"]

summary.sort_values("diff_correct_minus_wrong", ascending=False)

,correct_mean,wrong_mean,diff_correct_minus_wrong
DtRes,0.674419,0.387097,0.287322
AktAcc,0.395349,0.145161,0.250188
DtDTAM,0.023256,0.016129,0.007127
AktState,0.046512,0.048387,-0.001875
AktAch,0.674419,0.693548,-0.019130
DtUniv,0.046512,0.080645,-0.034134
AktAct,0.046512,0.177419,-0.130908
DtExp,0.279070,0.483871,-0.204801


In [7]:
df = filter_preds(
    rfcX,
    language="cmn",
    paradigm="le_guo",
    gold="le",
    features={"DtRes": 1}
)
save_csv(df[REPORT_COLS], "rfc_le_res1")

Saved: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Bible Data/Error Analysis/Reports/20260122_1659/rfc_le_res1.csv (53 rows)
